In [0]:
import pyspark.sql.functions as f
import pyspark.sql.types as t
from pyspark.sql.window import Window


In [0]:
spark.conf.get('spark.sql.files.maxPartitionBytes')

In [0]:
spark.conf.get('spark.sql.shuffle.partitions')


In [0]:
spark.conf.get('spark.sql.adaptive.enabled')

In [0]:
# Try to get all accessible configurations in Serverless
try:
    all_configs = spark.conf.getAll()
    print(f"Total accessible configs: {len(all_configs)}")
    print("\nAvailable configurations:")
    print("-" * 80)
    for key, value in sorted(all_configs.items()):
        print(f"{key}: {value}")
except Exception as e:
    print(f"Error getting all configs: {e}")
    print("\nTrying individual common configs...")
    
    # List of commonly needed configs to check
    common_configs = [
        'spark.sql.files.maxPartitionBytes',
        'spark.sql.shuffle.partitions',
        'spark.sql.adaptive.enabled',
        'spark.sql.adaptive.coalescePartitions.enabled',
        'spark.databricks.delta.optimizeWrite.enabled',
        'spark.databricks.delta.autoCompact.enabled',
    ]
    
    for conf_key in common_configs:
        try:
            value = spark.conf.get(conf_key)
            print(f"✓ {conf_key}: {value}")
        except Exception as ex:
            print(f"✗ {conf_key}: Not accessible - {str(ex)}")

In [0]:
df = spark.read.table('lk_test_catalog.iceberg_schema.people_iceberg_tbl')
display(df.limit(100))

In [0]:
df1 = df.groupBy('city').agg(f.countDistinct('id').alias('id_count'))
display(df1)

In [0]:
%sql
DESCRIBE DETAIL samples.bakehouse.sales_customers;

In [0]:
%sql
DESCRIBE FORMATTED samples.bakehouse.sales_customers;

In [0]:
cust_df = spark.read.table('samples.bakehouse.sales_customers')
transaction_df = spark.read.table('samples.bakehouse.sales_transactions')

join_df = cust_df.join(transaction_df, on='customerID', how='inner')
display(join_df)


In [0]:
join_broadcast_df = cust_df.join(transaction_df.hint('broadcast'), on='customer_id', how='inner')
display(join_broadcast_df)